# 01 Notebook Objective

# Synova RAG - Retrieval Test

## Objective

Retrieve the most relevant documents from the FAISS vector database using semantic search.

# 02 Import Libraries

In [1]:
from pathlib import Path
import sys
import pickle

import faiss
import pandas as pd

from sentence_transformers import SentenceTransformer

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# 03 Paths

In [2]:
VECTOR_DB_DIR = PROJECT_ROOT / "vector_db"

# 04 Load Model

In [3]:
MODEL_NAME = "BAAI/bge-m3"

model = SentenceTransformer(
    MODEL_NAME
)

# 05 Load FAISS Index

In [4]:
index = faiss.read_index(
    str(VECTOR_DB_DIR / "synova.index")
)

with open(
    VECTOR_DB_DIR / "metadata.pkl",
    "rb"
) as f:

    metadata = pickle.load(f)

print(
    f"Vectors: {index.ntotal}"
)

Vectors: 156


# 06 Language Detection

In [5]:
import re


def detect_language(text):

    if re.search(r'[\u0600-\u06FF]', text):
        return "ar"

    return "en"

# 07 Retrieval Function

In [6]:
def retrieve(
    query,
    top_k=5
):

    language = detect_language(query)

    query_embedding = model.encode(

        [query],

        normalize_embeddings=True

    )

    scores, indices = index.search(
        query_embedding,
        top_k * 3
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):

        doc = metadata[idx]

        if doc["language"] == language:

            results.append({

                "title": doc["title"],

                "category": doc["category"],

                "section": doc["section"],

                "score": float(score),

                "text": doc["text"]

            })

        if len(results) == top_k:
            break

    return results

# 08 Tests

# English Test

In [7]:
docs = retrieve(
    "VPN connection problem"
)

docs

[{'title': 'VPN Connection and Troubleshooting',
  'category': 'technical',
  'section': 'Symptoms',
  'score': 0.7374593615531921,
  'text': 'VPN login fails\nConnection timeout\nRemote resources cannot be accessed\nVPN disconnects every few minutes\nUnable to reach company systems'},
 {'title': 'VPN Connection and Troubleshooting',
  'category': 'technical',
  'section': 'Common Issues',
  'score': 0.7292059063911438,
  'text': 'Unable to connect to VPN\nVPN disconnects frequently\nAuthentication failed\nVPN timeout\nSlow VPN connection\nVPN connected but no internet access'},
 {'title': 'VPN Connection and Troubleshooting',
  'category': 'technical',
  'section': 'Troubleshooting',
  'score': 0.7093493938446045,
  'text': 'Verify your internet connection.\nRestart the VPN client.\nCheck your username and password.\nReconnect after a few minutes.\nUpdate the VPN application.\nRestart your computer.\nContact IT if the problem continues.'},
 {'title': 'VPN Connection and Troubleshootin

# Arabic Test

In [8]:
docs = retrieve(
    "نسيت كلمة المرور"
)

docs

[{'title': 'إعادة تعيين كلمة المرور والوصول إلى الحساب',
  'category': 'technical',
  'section': 'الأسئلة الشائعة',
  'score': 0.6566771864891052,
  'text': "{'question': 'نسيت كلمة المرور، ماذا أفعل؟', 'answer': 'استخدم بوابة إعادة تعيين كلمة المرور أو تواصل مع قسم الدعم الفني.'}\n{'question': 'تم قفل حسابي، كيف أستعيده؟', 'answer': 'انتظر حتى تنتهي مدة القفل أو اطلب من مسؤول النظام إعادة فتح الحساب.'}\n{'question': 'لم يصلني رمز التحقق (MFA)، ماذا أفعل؟', 'answer': 'تحقق من اتصال الإنترنت وتأكد من أن جهاز التحقق المسجل يعمل بشكل صحيح.'}"},
 {'title': 'إعادة تعيين كلمة المرور والوصول إلى الحساب',
  'category': 'technical',
  'section': 'المشكلات الشائعة',
  'score': 0.6419583559036255,
  'text': 'نسيان كلمة المرور\nقفل الحساب\nاسم مستخدم أو كلمة مرور غير صحيحة\nانتهاء صلاحية كلمة المرور\nالعودة المتكررة إلى صفحة تسجيل الدخول\nعدم وصول رمز التحقق\nانتهاء صلاحية رمز OTP\nتعطيل الحساب'},
 {'title': 'إعادة تعيين كلمة المرور والوصول إلى الحساب',
  'category': 'technical',
  'section': 'خطو

# HR Test

In [9]:
docs = retrieve(
    "How can I apply for annual leave?"
)

docs

[{'title': 'Employee Leave Policy',
  'category': 'hr',
  'section': 'FAQ',
  'score': 0.7476567029953003,
  'text': "{'question': 'How do I apply for annual leave?', 'answer': 'Submit a leave request through the HR system and wait for manager approval.'}\n{'question': 'Can I cancel my leave request?', 'answer': 'Yes, if it has not been approved yet or according to company policy.'}\n{'question': 'Do I need a medical report for sick leave?', 'answer': 'Depending on company policy, a medical certificate may be required.'}"},
 {'title': 'Employee Leave Policy',
  'category': 'hr',
  'section': 'Eligibility',
  'score': 0.6438458561897278,
  'text': 'Employees are eligible for annual leave according to company policy.\nSick leave requires supporting medical documentation when applicable.\nEmergency leave may require manager approval.'},
 {'title': 'Employee Leave Policy',
  'category': 'hr',
  'section': 'Leave Types',
  'score': 0.6123102903366089,
  'text': 'Annual Leave\nSick Leave\nEm

# HR Arabic Test

In [10]:
docs = retrieve(
    "كيف أقدم طلب إجازة؟"
)

docs

[{'title': 'سياسة الإجازات',
  'category': 'hr',
  'section': 'الأسئلة الشائعة',
  'score': 0.6964597702026367,
  'text': "{'question': 'كيف أطلب إجازة سنوية؟', 'answer': 'قدّم طلبًا عبر نظام الموارد البشرية وانتظر موافقة المدير.'}\n{'question': 'هل يمكنني إلغاء طلب الإجازة؟', 'answer': 'نعم، إذا لم تتم الموافقة عليه أو حسب سياسة الشركة.'}\n{'question': 'هل أحتاج إلى تقرير طبي للإجازة المرضية؟', 'answer': 'قد يتطلب الأمر تقريرًا طبيًا حسب سياسة الشركة.'}"},
 {'title': 'سياسة الإجازات',
  'category': 'hr',
  'section': 'أفضل الممارسات',
  'score': 0.6895648241043091,
  'text': 'قدّم طلب الإجازة مبكرًا.\nتحقق من رصيد الإجازات قبل تقديم الطلب.\nأبلغ مديرك بأي غياب طارئ.'},
 {'title': 'سياسة الإجازات',
  'category': 'hr',
  'section': 'خطوات طلب الإجازة',
  'score': 0.6887149810791016,
  'text': 'قدّم طلب الإجازة عبر نظام الموارد البشرية.\nحدد نوع الإجازة.\nحدد تاريخ البداية والنهاية.\nأرفق المستندات المطلوبة عند الحاجة.\nانتظر موافقة المدير قبل بدء الإجازة.'},
 {'title': 'سياسة الإجازات',